In [ ]:

#@title 1. Configuración del Entorno
import os

# Clonar el repositorio si no existe
if not os.path.exists("Signal-Analysis-Vol-I"):
    !git clone https://github.com/ecundir/Signal-Analysis-Vol-I.git

# Cambiar a la carpeta del capítulo
%cd Signal-Analysis-Vol-I/capitulo_4
print("✅ Entorno configurado correctamente.")


Para que tu laboratorio en Google Colab sea realmente efectivo, te sugiero seguir estas indicaciones durante la ejecución:

primero, observa la señal ideal y nota cómo sus bordes son instantáneos (verticales); luego, desplaza el control de frecuencia de corte (fc​) lentamente hacia la izquierda y fíjate en el gráfico del espectro para identificar el momento exacto en que desaparecen los armónicos impares (los "puntos rojos"). Presta especial atención a la gráfica de error instantáneo, ya que verás que los picos de error coinciden siempre con los flancos de la onda; si logras que el error se mantenga bajo durante el centro del pulso, la señal aún es "decodificable", pero si el redondeo es tan severo que el error invade todo el ciclo, habrás alcanzado el límite físico del canal. Por último, intenta encontrar la fc​ mínima necesaria para que la señal aún conserve su forma "achatada" característica antes de convertirse en una simple onda senoidal pura.

In [ ]:
# @title 🛠️ Laboratorio Vivo : "Espectro y Filtrado" { run: "auto" }
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import interact, FloatSlider, Layout

def laboratorio_completo(fc=20.0):
    # --- 1. PARÁMETROS Y SEÑAL IDEAL ---
    fs = 2000  # Mayor frecuencia de muestreo para ver mejor los armónicos
    t = np.linspace(0, 1, fs, endpoint=False)
    f_señal = 5
    x_ideal = signal.square(2 * np.pi * f_señal * t)

    # --- 2. FILTRADO (EL CANAL) ---
    sos = signal.butter(6, fc, 'low', fs=fs, output='sos')
    x_filtrada = signal.sosfilt(sos, x_ideal)
    error = x_ideal - x_filtrada

    # --- 3. ANÁLISIS DE FRECUENCIA (FFT) ---
    def obtener_fft(s, fs):
        n = len(s)
        freqs = np.fft.fftfreq(n, 1/fs)
        mag = np.abs(np.fft.fft(s)) / n
        return freqs[:n//2], mag[:n//2]

    fr_ideal, mag_ideal = obtener_fft(x_ideal, fs)
    fr_filt, mag_filt = obtener_fft(x_filtrada, fs)

    # --- 4. VISUALIZACIÓN ---
    fig = plt.figure(figsize=(12, 12))
    grid = plt.GridSpec(3, 1, hspace=0.4)

    # A. Gráfico de Tiempo (Señales)
    ax1 = fig.add_subplot(grid[0, 0])
    ax1.plot(t, x_ideal, 'k--', alpha=0.3, label='Ideal')
    ax1.plot(t, x_filtrada, 'r', label=f'Filtrada (fc={fc}Hz)', linewidth=2)
    ax1.set_title("Dominio del Tiempo: Deformación de la Onda", fontweight='bold')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.2)

    # B. Gráfico de Tiempo (Error)
    ax2 = fig.add_subplot(grid[1, 0])
    ax2.fill_between(t, 0, error, color='orange', alpha=0.4)
    ax2.plot(t, error, 'orange', linewidth=1)
    ax2.set_title("Error Instantáneo: e(t)", fontweight='bold')
    ax2.set_ylabel("Amplitud")
    ax2.grid(True, alpha=0.2)

    # C. Gráfico de Frecuencia (Espectro)
    ax3 = fig.add_subplot(grid[2, 0])
    ax3.stem(fr_ideal[:100], mag_ideal[:100], linefmt='k--', markerfmt='ko', label='Espectro Ideal', basefmt=" ")
    ax3.stem(fr_filt[:100], mag_filt[:100], linefmt='r-', markerfmt='ro', label='Espectro Filtrado', basefmt=" ")
    ax3.axvline(fc, color='blue', linestyle=':', label='Corte del Filtro (fc)')
    ax3.set_title("Dominio de la Frecuencia: Pérdida de Armónicos", fontweight='bold')
    ax3.set_xlabel("Frecuencia (Hz)")
    ax3.set_ylabel("Magnitud")
    ax3.legend()
    ax3.set_xlim(0, 150)
    ax3.grid(True, alpha=0.2)

    plt.show()

interact(laboratorio_completo,
         fc=FloatSlider(value=40.0, min=5.0, max=150.0, step=5.0,
                         description='Corte (fc):', layout=Layout(width='80%')));